In [1]:

import numpy as np

from tslearn.datasets import UCR_UEA_datasets
import torch
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from uni2ts.model.moirai import MoiraiModule
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
#import seaborn as sns
import copy
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
from encoder import MoiraiEncoder
from sklearn.model_selection import train_test_split

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Utils

In [2]:
PATCH_SIZES = [8, 16, 32, 64]

SIZE = "small"
DEVICE = "cuda"
NUM_VARS = 6

DO_MASK = True
KEEP_MASK_EMBEDDING = True

In [3]:
def preprocess_data(
    data: np.ndarray,
    *,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
):
    """
    data: np.ndarray of shape (N, T, V) = (n_individual, time, variate)
    Assumes NO missing values and NO padding in the raw data.
    """

    if not isinstance(data, np.ndarray):
        raise TypeError("data must be a numpy.ndarray")
    if data.ndim != 3:
        raise ValueError(f"Expected shape (N,T,V), got {data.shape}")

    N, T, V = data.shape

    # (N,T,V)
    past_target = torch.as_tensor(data, dtype=dtype, device=device)

    # observed mask: (N,T,V) all True no missing values
    past_observed_target = torch.ones((N, T, V), dtype=torch.bool, device=device)

    # padding mask: (N,T) all if no padding
    past_is_pad = torch.zeros((N, T), dtype=torch.bool, device=device)

    return past_target, past_observed_target, past_is_pad

In [4]:
def apply_pooling_pt(Z_tensor, method, num_vars=NUM_VARS):
    N, S, F = Z_tensor.shape
    P = S // num_vars # Calcul automatique du nombre de patches par variable
    
    # On reshape le tenseur pour séparer les Variables et les Patches
    # Forme résultante : (Batch, Variables, Patches, Features)
    Z_reshaped = Z_tensor.view(N, num_vars, P, F)
    
    # Basique et Global
    if method == "flatten":
        return Z_tensor.reshape(N, -1)
        
    elif method == "global_mean":
        return Z_tensor.mean(dim=1)
        
    elif method == "global_max":
        return Z_tensor.max(dim=1).values
        
    elif method == "global_min":
        return Z_tensor.min(dim=1).values
    
    elif method == "global_mean_max_min":
        return torch.cat([
            Z_tensor.mean(dim=1),
            Z_tensor.max(dim=1).values,
            Z_tensor.min(dim=1).values
        ], dim=1)

    # Pooling sur les Patches (on garde les variables distinctes) ---
    # Réduction sur la dimension 2 (Patches). Résultat : (N, num_vars, F), puis on aplatit
    elif method == "mean_over_patches":
        return Z_reshaped.mean(dim=2).reshape(N, -1)
        
    elif method == "max_over_patches":
        return Z_reshaped.max(dim=2).values.reshape(N, -1)
        
    elif method == "min_over_patches":
        return Z_reshaped.min(dim=2).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_patches":
        p_mean = Z_reshaped.mean(dim=2).reshape(N, -1)
        p_max  = Z_reshaped.max(dim=2).values.reshape(N, -1)
        p_min  = Z_reshaped.min(dim=2).values.reshape(N, -1)
        return torch.cat([p_mean, p_max, p_min], dim=1)

    # Pooling sur les Variables (on synchronise les patches entre variables) ---
    # Réduction sur la dimension 1 (Variables). Résultat : (N, P, F), puis on aplatit
    elif method == "mean_over_variables":
        return Z_reshaped.mean(dim=1).reshape(N, -1)
        
    elif method == "max_over_variables":
        return Z_reshaped.max(dim=1).values.reshape(N, -1)
        
    elif method == "min_over_variables":
        return Z_reshaped.min(dim=1).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_variables":
        v_mean = Z_reshaped.mean(dim=1).reshape(N, -1)
        v_max  = Z_reshaped.max(dim=1).values.reshape(N, -1)
        v_min  = Z_reshaped.min(dim=1).values.reshape(N, -1)
        return torch.cat([v_mean, v_max, v_min], dim=1)

    else:
        raise ValueError(f"Method {method} unknow")

pooling_methods = [
    "flatten", 
    "global_mean", "global_max", "global_min", "global_mean_max_min",
    "mean_over_patches", "max_over_patches", "min_over_patches", "mean_max_min_over_patches",
    "mean_over_variables", "max_over_variables", "min_over_variables", "mean_max_min_over_variables"
]

In [5]:
alphas_to_test = [0.1,1,10,30,100,300,1000,10000]

models = {
    'Ridge': RidgeClassifierCV(alphas=alphas_to_test, cv=3),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# Data

Import

In [6]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

NUM_CLASSES = len(torch.unique(y_train_tensor))

Prepare data

In [7]:
X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = (
    preprocess_data(X_train, device=DEVICE)
)

X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = (
    preprocess_data(X_test, device=DEVICE)
)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

num_classes = len(torch.unique(y_train_tensor))
NUM_VARS = 6

print(X_train_target_tensor.shape)
print(X_train_is_target_observed.shape)
print(X_train_is_target_padded.shape)

torch.Size([2459, 36, 6])
torch.Size([2459, 36, 6])
torch.Size([2459, 36])


Creat Z_train and Z_test for each patch size

In [8]:
Z_train_patch = {}
Z_test_patch = {}
for patch in PATCH_SIZES:
    
    # 1. Définir les paramètres et l'encodeur pour la taille de patch actuelle
    MODEL_PARAMS = {
        "patch_size": patch,
        "num_samples": 100,
        "target_dim": 6,
        "feat_dynamic_real_dim": 0,
        "past_feat_dynamic_real_dim": 0,
    }
    if DO_MASK == True:
        prediction_length = patch
    else:
        prediction_length = 0
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=prediction_length,
        context_length=36,
        **MODEL_PARAMS,
    )
    encoder.eval()
    encoder.to(DEVICE)
    
    # 2. Extraire les embeddings (Z_train et Z_test)
    with torch.no_grad():
        Z_train = encoder(
            past_target=X_train_target_tensor,
            past_observed_target=X_train_is_target_observed,
            past_is_pad=X_train_is_target_padded,
        )
        Z_test = encoder(
            past_target=X_test_target_tensor,
            past_observed_target=X_test_is_target_observed,
            past_is_pad=X_test_is_target_padded,
        )
        
    if DO_MASK and not KEEP_MASK_EMBEDDING:
        N, S, F = Z_train.shape
        P = S // NUM_VARS  # Nombre de patchs par variable (Contexte + 1 Masque)
        
        # On reshape pour séparer les variables et les patchs : (Batch, 6 vars, Patchs, Features)
        Z_train_reshaped = Z_train.view(N, NUM_VARS, P, F)
        Z_test_reshaped = Z_test.view(Z_test.shape[0], NUM_VARS, P, F) # Attention à bien récupérer N_test = Z_test.shape[0]
        
        # On slice pour enlever le dernier patch (:-1) sur la dimension des patchs (dim=2)
        Z_train_no_mask = Z_train_reshaped[:, :, :-1, :]
        Z_test_no_mask = Z_test_reshaped[:, :, :-1, :]
        
        # On remet sous la forme attendue (Batch, Séquence_sans_masques, Features)
        Z_train = Z_train_no_mask.reshape(N, -1, F)
        Z_test = Z_test_no_mask.reshape(Z_test.shape[0], -1, F)



    Z_train_patch[patch] = Z_train
    Z_test_patch[patch] = Z_test

In [9]:

print(f"{'Patch Size':<12} | {'Train Shape':<25} | {'Test Shape':<25}")
for patch in PATCH_SIZES:
    train_tensor = Z_train_patch[patch]
    test_tensor = Z_test_patch[patch]
    print(f"{patch:<12} | {str(list(train_tensor.shape)):<25} | {str(list(test_tensor.shape)):<25}")

Patch Size   | Train Shape               | Test Shape               
8            | [2459, 36, 384]           | [2466, 36, 384]          
16           | [2459, 24, 384]           | [2466, 24, 384]          
32           | [2459, 18, 384]           | [2466, 18, 384]          
64           | [2459, 12, 384]           | [2466, 12, 384]          


# Basic pooling with logistic regression and random forest

## Single patch size pooling

We started by applying simple pooling techniques (max, min, or mean) and comparing their performance across various patch sizes. Specifically, we evaluated whether to apply this pooling globally across all patches, locally within a single series, or across corresponding patches from all series.

### Training

We creat a dataframe to stock the results and we fill it

In [10]:
results_dfs = {
    model_name: pd.DataFrame(index=pooling_methods, columns=PATCH_SIZES)
    for model_name in models.keys()
}
for df in results_dfs.values():
    df.index.name = "Pooling \\ Patch"

We train

In [11]:
for patch in PATCH_SIZES:
    Z_train = Z_train_patch[patch].to(DEVICE)
    Z_test = Z_test_patch[patch].to(DEVICE)
    
    for pool in pooling_methods:
        X_train_pool = apply_pooling_pt(Z_train, pool).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool).cpu().numpy()

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_pool)
        X_test_scaled = scaler.transform(X_test_pool)
        
        for model_name, clf in models.items():
            clf.fit(X_train_scaled, y_train)
            
            y_pred = clf.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_pred)
            
            results_dfs[model_name].loc[pool, patch] = round(acc, 3)


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=2.96243e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=2.99794e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=2.95261e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditio

### Results

In [12]:
for model_name, df in results_dfs.items():
    print(f"RÉSULTATS POUR : {model_name.upper()}")
    display(df.astype(float))
    
    best_acc = df.max().max()
    best_pool, best_patch = df.astype(float).stack().idxmax()
    print(f"Meilleure combinaison = Patch: {best_patch} | Pooling: '{best_pool}' | Acc: {best_acc:.3f}\n")

RÉSULTATS POUR : RIDGE


,8,16,32,64
Pooling \ Patch,,,,
flatten,0.608,0.625,0.620,0.633
global_mean,0.586,0.590,0.583,0.598
global_max,0.544,0.552,0.536,0.578
global_min,0.545,0.553,0.538,0.565
global_mean_max_min,0.577,0.581,0.577,0.594
mean_over_patches,0.617,0.623,0.619,0.636
max_over_patches,0.590,0.610,0.601,0.632
min_over_patches,0.589,0.605,0.603,0.632
mean_max_min_over_patches,0.611,0.631,0.626,0.633


Meilleure combinaison = Patch: 64 | Pooling: 'mean_over_patches' | Acc: 0.636

RÉSULTATS POUR : RANDOMFOREST


,8,16,32,64
Pooling \ Patch,,,,
flatten,0.567,0.553,0.551,0.592
global_mean,0.574,0.576,0.559,0.579
global_max,0.546,0.522,0.521,0.564
global_min,0.523,0.528,0.515,0.565
global_mean_max_min,0.571,0.573,0.547,0.580
mean_over_patches,0.591,0.578,0.559,0.602
max_over_patches,0.565,0.566,0.552,0.597
min_over_patches,0.559,0.559,0.552,0.597
mean_max_min_over_patches,0.581,0.573,0.564,0.599


Meilleure combinaison = Patch: 64 | Pooling: 'mean_over_patches' | Acc: 0.602



df_ridge_plot = df_results_ridge.astype(float)
df_rf_plot = df_results_rf.astype(float)


fig, axes = plt.subplots(1, 2, figsize=(20, 10), sharey=True)

cmap = "YlGnBu"

sns.heatmap(df_ridge_plot, 
            annot=True,
            fmt=".3f",
            cmap=cmap,
            linewidths=0.5,
            ax=axes[0],
            cbar_kws={'label': 'Accuracy'})
axes[0].set_title("Single Patch Pooling - Accuracy (Ridge)", fontsize=16, fontweight='bold', pad=15)
axes[0].set_xlabel("Patch Size", fontsize=14)
axes[0].set_ylabel("Pooling Method", fontsize=14)
axes[0].tick_params(axis='both', which='major', labelsize=12)

sns.heatmap(df_rf_plot, 
            annot=True, 
            fmt=".3f", 
            cmap=cmap, 
            linewidths=0.5, 
            ax=axes[1], 
            cbar_kws={'label': 'Accuracy'})
axes[1].set_title("Single Patch Pooling - Accuracy (Random Forest)", fontsize=16, fontweight='bold', pad=15)
axes[1].set_xlabel("Patch Size", fontsize=14)
axes[1].tick_params(axis='both', which='major', labelsize=12)

plt.tight_layout()

plt.show()

Pooling over patches rather than series produces the highest accuracy.

## Multi Patch pooling

### Training

In [13]:
df_multiscale_results = pd.DataFrame(index=pooling_methods, columns=models.keys())
df_multiscale_results.index.name = "Pooling Method"

for pool in pooling_methods:
    X_train_multi = []
    X_test_multi = []
    
    for patch in PATCH_SIZES:
        Z_train = Z_train_patch[patch].to(DEVICE)
        Z_test = Z_test_patch[patch].to(DEVICE)
        
        X_train_pool = apply_pooling_pt(Z_train, pool).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool).cpu().numpy()
        
        X_train_multi.append(X_train_pool)
        X_test_multi.append(X_test_pool)
        
    X_train_combined = np.concatenate(X_train_multi, axis=1)
    X_test_combined = np.concatenate(X_test_multi, axis=1)
    
    scaler = StandardScaler()
    X_train_combined_scaled = scaler.fit_transform(X_train_combined)
    X_test_combined_scaled = scaler.transform(X_test_combined)

    for model_name, clf in models.items():
        clf.fit(X_train_combined_scaled, y_train)
        
        y_pred = clf.predict(X_test_combined_scaled)
        acc = accuracy_score(y_test, y_pred)
        
        df_multiscale_results.loc[pool, model_name] = round(acc, 3)

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.40241e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.42041e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.37498e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: Ill-conditio

### Results

In [14]:
display(df_multiscale_results.astype(float))

best_acc = df_multiscale_results.max().max()
best_pool = df_multiscale_results.max(axis=1).idxmax()
best_model = df_multiscale_results.max(axis=0).idxmax()

print("MEILLEURE COMBINAISON MULTI-ÉCHELLE :")
print(f"Modèle : '{best_model}' | Pooling : '{best_pool}'")
print(f"Précision maximale : {best_acc:.3f}")

,Ridge,RandomForest
Pooling Method,,
flatten,0.649,0.580
global_mean,0.623,0.594
global_max,0.591,0.564
global_min,0.587,0.562
global_mean_max_min,0.612,0.590
mean_over_patches,0.658,0.608
max_over_patches,0.659,0.591
min_over_patches,0.650,0.596
mean_max_min_over_patches,0.664,0.599


MEILLEURE COMBINAISON MULTI-ÉCHELLE :
Modèle : 'Ridge' | Pooling : 'mean_max_min_over_patches'
Précision maximale : 0.664


## Conclusion

pool over patch -> good
16 -> (max, mean, min) > flatten -> position not importante
multi patch -> good

# Advanced Pooling

## Attention Pooling

In [15]:
patch_sizes = [8, 16, 32, 64]
NUM_VARS = 6

patches_counts = [Z_train_patch[p].shape[1] // NUM_VARS for p in patch_sizes]
print(f"Nomber of patch for each scale : {patches_counts}")

Z_train_concat = torch.cat([Z_train_patch[p].to(DEVICE) for p in patch_sizes], dim=1)
Z_test_concat  = torch.cat([Z_test_patch[p].to(DEVICE) for p in patch_sizes], dim=1)

print(f"Shape Z_train: {Z_train_concat.shape}")

Nomber of patch for each scale : [6, 4, 3, 2]
Shape Z_train: torch.Size([2459, 90, 384])


### Validation pipeline
https://arxiv.org/abs/2502.15637
on valide sur la loss ( pass acc pourune meilleur généralisation)

In [16]:
PATCH = 16
BATCH_SIZE = 256

Z_train_full = Z_train_patch[PATCH].cpu().numpy()
y_train_full = y_train_tensor.cpu().numpy()

#80% Train / 20% Validation
Z_train_split, Z_val_split, y_train_split, y_val_split = train_test_split(
    Z_train_full, 
    y_train_full, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_train_full
)

Z_train_t = torch.tensor(Z_train_split, device=DEVICE)
y_train_t = torch.tensor(y_train_split, dtype=torch.long, device=DEVICE)

Z_val_t = torch.tensor(Z_val_split, device=DEVICE)
y_val_t = torch.tensor(y_val_split, dtype=torch.long, device=DEVICE)

Z_test_t = Z_test_patch[PATCH].to(DEVICE)
y_test_t = y_test_tensor.to(DEVICE)

train_loader = DataLoader(TensorDataset(Z_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(Z_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(Z_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)

print(f"Length -> Train : {len(Z_train_t)} | Val : {len(Z_val_t)} | Test : {len(Z_test_t)}")

Length -> Train : 1967 | Val : 492 | Test : 2466


In [17]:
def train(
    model, 
    train_loader, 
    val_loader, 
    lr, 
    epochs=100, 
    weight_decay=0.005,
    device="cuda",
    verbose=False
):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    patience = 50
    epochs_no_improve = 0
    best_avg_val_loss = np.inf
    best_model_weights = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs):
        model.train()
        for batch_z, batch_y in train_loader:
            batch_z, batch_y = batch_z.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            logits = model(batch_z)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
        

        model.eval()

        """
        correct, total = 0, 0
        with torch.no_grad():
            for batch_z, batch_y in val_loader:
                batch_z, batch_y = batch_z.to(device), batch_y.to(device)
                logits = model(batch_z)
                predictions = torch.argmax(logits, dim=-1)
                correct += (predictions == batch_y).sum().item()
                total += batch_y.size(0)
                
        val_acc = correct / total
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_weights = copy.deepcopy(model.state_dict())
        """
        model.eval()
        total_val_loss = 0.0
        correct, total = 0, 0
        
        with torch.no_grad():
            for batch_z, batch_y in val_loader:
                batch_z, batch_y = batch_z.to(device), batch_y.to(device)
                
                logits = model(batch_z)
                
                loss = criterion(logits, batch_y)
                total_val_loss += loss.item() * batch_y.size(0)
                
                predictions = torch.argmax(logits, dim=-1)
                correct += (predictions == batch_y).sum().item()
                total += batch_y.size(0)
                
        avg_val_loss = total_val_loss / total
        #val_acc = correct / total
        #tab to early_stopping
        if avg_val_loss < best_avg_val_loss:
            best_avg_val_loss =  avg_val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            print('early_stop', epoch)
            break
            
    model.load_state_dict(best_model_weights)
    #return best_val_acc, model
    return best_avg_val_loss, model

In [18]:
def grid_search_attention_model(
    model_class, 
    model_kwargs, 
    train_loader, 
    val_loader, 
    test_loader, 
    device="cuda"
):
    lr_grid = [1e-4]
    wd_grid = [0.0, 0.05, 0.1, 0.2] 
    
    
    best_overall_val_acc = 0.0
    best_overall_val_loss = np.inf
    best_lr = None
    best_wd = None
    best_model = None
    for wd in wd_grid:
        for lr in lr_grid:
            print(f"LR = {lr} | WD = {wd}")
            
            model = model_class(**model_kwargs).to(device)

            """
            val_acc, trained_model = train(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                lr=lr,
                epochs=200,
                weight_decay=wd,
                device=device
            )
            
            print(f" -> Validation Accuracy : {val_acc:.4f}")
            
            if val_acc > best_overall_val_acc:
                best_overall_val_acc = val_acc
                best_lr = lr
                best_wd = wd
                best_model = copy.deepcopy(trained_model)
            """
            val_loss, trained_model = train(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                lr=lr,
                epochs=500,
                weight_decay=wd,
                device=device
            )
            
            print(f" -> Validation loss : {val_loss:.4f}")
            
            if val_loss < best_overall_val_loss:
                best_overall_val_loss = val_loss
                best_lr = lr
                best_wd = wd
                best_model = copy.deepcopy(trained_model)     
            
    #print(f"\n Best LR : {best_lr} | Best WD : {best_wd} (Val Acc: {best_overall_val_acc:.4f})")
    print(f"\n Best LR : {best_lr} | Best WD : {best_wd} (Val Avg Loss: {best_overall_val_loss:.4f})")
    
    best_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch_z, batch_y in test_loader:
            batch_z, batch_y = batch_z.to(device), batch_y.to(device)
            logits = best_model(batch_z)
            predictions = torch.argmax(logits, dim=-1)
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)
            
    test_acc = correct / total
    print(f"Test Accuracy : {test_acc:.4f}\n")
    
    return best_model, test_acc

### Models

In [19]:
class SingleScaleAttentionClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, in_features=384, mode="shared_context"):
        """
        mode: 
          - "shared_context": 1 seul vecteur contexte pour les 6 variables.
          - "independent_context": 6 vecteurs contextes distincts (1 par variable).
        """
        super().__init__()
        self.num_vars = num_vars
        self.mode = mode

        if mode == "shared_context":
            self.context = nn.Parameter(torch.randn(in_features) * 0.1)
            
        elif mode == "independent_context":
            self.context = nn.Parameter(torch.randn(num_vars, in_features) * 0.1)
            
        self.attn_dropout = nn.Dropout(0.1)
        self.dropout = nn.Dropout(0.1)
        
        linear_in = num_vars * in_features
        self.linear = nn.Linear(linear_in, num_classes)
        

    def forward(self, x):
        # x shape: (Batch, Séquence Totale, Features)
        B = x.shape[0]
        S = x.shape[1]
        F = x.shape[-1]
        
        P = S // self.num_vars

        x_reshaped = x.view(B, self.num_vars, P, F)
        
        if self.mode == "shared_context":
            # x_reshaped: (B, V, P, F) | context: (F)
            scores = torch.matmul(x_reshaped, self.context) # Résultat: (B, V, P)
            
        elif self.mode == "independent_context":

            context_view = self.context.view(1, self.num_vars, 1, F)
            scores = (x_reshaped * context_view).sum(dim=-1) # Résultat: (B, V, P)
            
        weights = torch.softmax(scores, dim=2).unsqueeze(-1) # shape: (B, V, P, 1)
        weights = self.attn_dropout(weights)
        pooled_vars = (weights * x_reshaped).sum(dim=2) # shape: (B, V, F)
        
        pooled = pooled_vars.view(B, -1) # shape: (B, V * F)
        
        return self.linear(self.dropout(pooled))

In [20]:
class SingleScaleMultiHeadClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, in_features=384, num_heads=4, patch_mode="shared_context"):
        """
        mode: 
          - "shared_context": 1 seul token [CLS] (Query) partagé pour lire les 6 variables.
          - "independent_context": 6 tokens [CLS] distincts (1 spécialiste par variable).
        """
        super().__init__()
        self.num_vars = num_vars
        self.mode = patch_mode
        
        if self.mode == "shared_context":
            self.cls_token = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)
            
        elif self.mode == "independent_context":
            self.cls_token = nn.Parameter(torch.randn(num_vars, 1, in_features) * 0.02)

            
        self.mha = nn.MultiheadAttention(
            embed_dim=in_features,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True 
        )
            
        self.dropout = nn.Dropout(0.1)

        linear_in = num_vars * in_features
        self.linear = nn.Linear(linear_in, num_classes)


    def forward(self, x):
        B = x.shape[0]
        S = x.shape[1]
        F = x.shape[-1]

        P = S // self.num_vars

        x_reshaped = x.view(B, self.num_vars, P, F)
        

        kv = x_reshaped.view(B * self.num_vars, P, F)
        
        if self.mode == "shared_context":
            q = self.cls_token.expand(B * self.num_vars, -1, -1)
            
        elif self.mode == "independent_context":
            q_expanded = self.cls_token.unsqueeze(0).expand(B, -1, -1, -1)
            q = q_expanded.reshape(B * self.num_vars, 1, F)
            

        attn_output, attn_weights = self.mha(query=q, key=kv, value=kv)
        

        pooled_flat = attn_output.squeeze(1) # shape: (B * V, F)
        
        pooled_vars = pooled_flat.view(B, self.num_vars, F) # shape: (B, V, F)
        
        pooled = pooled_vars.view(B, -1) # shape: (B, V * F)
        

        return self.linear(self.dropout(pooled))

In [21]:
class SingleScaleMultiHeadClassifierPos(nn.Module):
    def __init__(self, num_vars, num_classes,num_patches = 3, in_features=384, num_heads=4, patch_mode="shared_context"):
        """
        mode: 
          - "shared_context": 1 seul token [CLS] (Query) partagé pour lire les 6 variables.
          - "independent_context": 6 tokens [CLS] distincts (1 spécialiste par variable).
        """
        super().__init__()
        self.num_vars = num_vars
        self.mode = patch_mode
        self.num_patches = num_patches
        
        if self.mode == "shared_context":
            self.cls_token = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)
            
        elif self.mode == "independent_context":
            self.cls_token = nn.Parameter(torch.randn(num_vars, 1, in_features) * 0.02)

        self.pos_embed = nn.Parameter(torch.randn(1, 1, num_patches, in_features) * 0.02)
            
        self.mha = nn.MultiheadAttention(
            embed_dim=in_features,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True 
        )
        self.dropout = nn.Dropout(0.1)

        linear_in = num_vars * in_features
        self.linear = nn.Linear(linear_in, num_classes)


    def forward(self, x):
        B = x.shape[0]
        S = x.shape[1]
        F = x.shape[-1]

        P = S // self.num_vars

        x_reshaped = x.view(B, self.num_vars, P, F)

        x_reshaped = x_reshaped + self.pos_embed
    
        kv = x_reshaped.view(B * self.num_vars, P, F)
        
        if self.mode == "shared_context":
            q = self.cls_token.expand(B * self.num_vars, -1, -1)
            
        elif self.mode == "independent_context":
            q_expanded = self.cls_token.unsqueeze(0).expand(B, -1, -1, -1)
            q = q_expanded.reshape(B * self.num_vars, 1, F)
            

        attn_output, attn_weights = self.mha(query=q, key=kv, value=kv)
        
        pooled_flat = attn_output.squeeze(1) # shape: (B * V, F)
        
        pooled_vars = pooled_flat.view(B, self.num_vars, F) # shape: (B, V, F)
        
        pooled = pooled_vars.view(B, -1) # shape: (B, V * F)
        

        return self.linear(self.dropout(pooled))

In [22]:
class HierarchicalMultiHeadClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, in_features=384, num_heads=4, patch_mode="independent_context"):
        super().__init__()
        self.num_vars = num_vars
        self.patch_mode = patch_mode
        

        if patch_mode == "shared_context":
            self.patch_cls = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)
        elif patch_mode == "independent_context":
            self.patch_cls = nn.Parameter(torch.randn(num_vars, 1, in_features) * 0.02)


        self.patch_mha = nn.MultiheadAttention(
            embed_dim=in_features, num_heads=num_heads, dropout=0.1, batch_first=True
        )


        self.var_cls = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)

        self.var_mha = nn.MultiheadAttention(
            embed_dim=in_features, num_heads=num_heads, dropout=0.1, batch_first=True
        )

        self.dropout = nn.Dropout(0.1)

        self.linear = nn.Linear(in_features, num_classes)


    def forward(self, x):
        B = x.shape[0]
        S = x.shape[1]
        F = x.shape[-1]
        
        P = S // self.num_vars
        
        x_reshaped = x.view(B, self.num_vars, P, F)
        kv_patch = x_reshaped.view(B * self.num_vars, P, F) # (B*V, P, F)
        
        if self.patch_mode == "shared_context":
            q_patch = self.patch_cls.expand(B * self.num_vars, -1, -1)
        elif self.patch_mode == "independent_context":
            q_expanded = self.patch_cls.unsqueeze(0).expand(B, -1, -1, -1)
            q_patch = q_expanded.reshape(B * self.num_vars, 1, F)
            
        patch_attn_out, _ = self.patch_mha(query=q_patch, key=kv_patch, value=kv_patch)

        var_reprs = patch_attn_out.squeeze(1).view(B, self.num_vars, F) # (B, V, F)
        

        kv_var = var_reprs # (B, V, F)
        

        q_var = self.var_cls.expand(B, -1, -1) # (B, 1, F)

        var_attn_out, _ = self.var_mha(query=q_var, key=kv_var, value=kv_var)

        global_repr = var_attn_out.squeeze(1) # (B, F)

        return self.linear(self.dropout(global_repr))

### Train

In [23]:
PATCH_SIZES_TO_TEST = [8, 16]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 8

model_names = [
    "SingleScale Attention", 
    "SingleScale MHA (Heads=8)", 
    "Hierarchical MHA (Heads=8)"
]

index = pd.MultiIndex.from_product([model_names, MODES], names=["Modèle", "Mode"])
df_results = pd.DataFrame(index=index, columns=PATCH_SIZES_TO_TEST)
df_results.columns.name = "Patch Size"

def get_dataloaders_for_patch(patch_size, batch_size=256):
    Z_train_full = Z_train_patch[patch_size].cpu().numpy()
    y_train_full = y_train_tensor.cpu().numpy()
    
    Z_train_split, Z_val_split, y_train_split, y_val_split = train_test_split(
        Z_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
    )
    
    Z_tr = torch.tensor(Z_train_split, device=DEVICE)
    y_tr = torch.tensor(y_train_split, dtype=torch.long, device=DEVICE)
    Z_va = torch.tensor(Z_val_split, device=DEVICE)
    y_va = torch.tensor(y_val_split, dtype=torch.long, device=DEVICE)
    Z_te = Z_test_patch[patch_size].to(DEVICE)
    y_te = y_test_tensor.to(DEVICE)
    
    tr_loader = DataLoader(TensorDataset(Z_tr, y_tr), batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(TensorDataset(Z_va, y_va), batch_size=batch_size, shuffle=False)
    te_loader = DataLoader(TensorDataset(Z_te, y_te), batch_size=batch_size, shuffle=False)
    
    return tr_loader, va_loader, te_loader

for patch in PATCH_SIZES_TO_TEST:
    tr_loader, va_loader, te_loader = get_dataloaders_for_patch(patch)
    
    for mode in MODES:
        print(f"PATCH: {patch} | MODE: {mode}")
        _, test_acc = grid_search_attention_model(
            model_class=SingleScaleAttentionClassifier,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_results.loc[("SingleScale Attention", mode), patch] = test_acc
        _, test_acc = grid_search_attention_model(
            model_class=SingleScaleMultiHeadClassifier,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_results.loc[("SingleScale MHA (Heads=8)", mode), patch] = test_acc

        _, test_acc = grid_search_attention_model(
            model_class=HierarchicalMultiHeadClassifier,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_results.loc[("Hierarchical MHA (Heads=8)", mode), patch] = test_acc

PATCH: 8 | MODE: shared_context
LR = 0.0001 | WD = 0.0
early_stop 293
 -> Validation loss : 1.2282
LR = 0.0001 | WD = 0.05
early_stop 376
 -> Validation loss : 1.2161
LR = 0.0001 | WD = 0.1
early_stop 422
 -> Validation loss : 1.2155
LR = 0.0001 | WD = 0.2
early_stop 282
 -> Validation loss : 1.2871

 Best LR : 0.0001 | Best WD : 0.1 (Val Avg Loss: 1.2155)
Test Accuracy : 0.5965

LR = 0.0001 | WD = 0.0
early_stop 110
 -> Validation loss : 1.1635
LR = 0.0001 | WD = 0.05
early_stop 122
 -> Validation loss : 1.1467
LR = 0.0001 | WD = 0.1
early_stop 113
 -> Validation loss : 1.1324
LR = 0.0001 | WD = 0.2
early_stop 104
 -> Validation loss : 1.1424

 Best LR : 0.0001 | Best WD : 0.1 (Val Avg Loss: 1.1324)
Test Accuracy : 0.6111

LR = 0.0001 | WD = 0.0
early_stop 131
 -> Validation loss : 1.2916
LR = 0.0001 | WD = 0.05
early_stop 142
 -> Validation loss : 1.2572
LR = 0.0001 | WD = 0.1
early_stop 135
 -> Validation loss : 1.2708
LR = 0.0001 | WD = 0.2
early_stop 135
 -> Validation loss : 1.27

### Results

In [24]:
display(df_results.astype(float).round(4))

Patch Size                                          8       16
Modèle                     Mode                               
SingleScale Attention      shared_context       0.5965  0.6034
                           independent_context  0.5912  0.6030
SingleScale MHA (Heads=8)  shared_context       0.6111  0.6172
                           independent_context  0.6148  0.6253
Hierarchical MHA (Heads=8) shared_context       0.5791  0.5835
                           independent_context  0.5957  0.5864

In [25]:
PATCH = 16
MODES = ["shared_context", "independent_context"]
HEADS_TO_TEST = [16, 32, 64, 128, 384] 

df_heads_results = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_results.index.name = "Mode"
df_heads_results.columns.name = f"Num Heads (Patch {PATCH})"

tr_loader, va_loader, te_loader = get_dataloaders_for_patch(PATCH)

for mode in MODES:
    for heads in HEADS_TO_TEST:
        print(f"MODE: {mode} | HEADS: {heads}")
        
        _, test_acc = grid_search_attention_model(
            model_class=SingleScaleMultiHeadClassifier,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        
        df_heads_results.loc[mode, heads] = test_acc

MODE: shared_context | HEADS: 16
LR = 0.0001 | WD = 0.0
early_stop 111
 -> Validation loss : 1.1203
LR = 0.0001 | WD = 0.05
early_stop 105
 -> Validation loss : 1.1161
LR = 0.0001 | WD = 0.1
early_stop 101
 -> Validation loss : 1.1370
LR = 0.0001 | WD = 0.2
early_stop 101
 -> Validation loss : 1.1201

 Best LR : 0.0001 | Best WD : 0.05 (Val Avg Loss: 1.1161)
Test Accuracy : 0.6249

MODE: shared_context | HEADS: 32
LR = 0.0001 | WD = 0.0
early_stop 103
 -> Validation loss : 1.1152
LR = 0.0001 | WD = 0.05
early_stop 111
 -> Validation loss : 1.1053
LR = 0.0001 | WD = 0.1
early_stop 107
 -> Validation loss : 1.1036
LR = 0.0001 | WD = 0.2
early_stop 103
 -> Validation loss : 1.1152

 Best LR : 0.0001 | Best WD : 0.1 (Val Avg Loss: 1.1036)
Test Accuracy : 0.6237

MODE: shared_context | HEADS: 64
LR = 0.0001 | WD = 0.0
early_stop 105
 -> Validation loss : 1.1071
LR = 0.0001 | WD = 0.05
early_stop 104
 -> Validation loss : 1.1089
LR = 0.0001 | WD = 0.1
early_stop 107
 -> Validation loss : 1.0

In [26]:
PATCH = 16
MODES = ["shared_context", "independent_context"]
HEADS_TO_TEST = [16, 32, 64, 128, 384] 

df_heads_pos_results = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_pos_results.index.name = "Mode"
df_heads_pos_results.columns.name = f"Num Heads (Patch {PATCH})"

tr_loader, va_loader, te_loader = get_dataloaders_for_patch(PATCH)

for mode in MODES:
    for heads in HEADS_TO_TEST:
        print(f"MODE: {mode} | HEADS: {heads}")
        
        _, test_acc = grid_search_attention_model(
            model_class=SingleScaleMultiHeadClassifierPos,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads, 'num_patches' : 4},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        
        df_heads_pos_results.loc[mode, heads] = test_acc

MODE: shared_context | HEADS: 16
LR = 0.0001 | WD = 0.0
early_stop 101
 -> Validation loss : 1.1156
LR = 0.0001 | WD = 0.05
early_stop 101
 -> Validation loss : 1.1298
LR = 0.0001 | WD = 0.1
early_stop 109
 -> Validation loss : 1.1220
LR = 0.0001 | WD = 0.2
early_stop 105
 -> Validation loss : 1.1225

 Best LR : 0.0001 | Best WD : 0.0 (Val Avg Loss: 1.1156)
Test Accuracy : 0.6192

MODE: shared_context | HEADS: 32
LR = 0.0001 | WD = 0.0
early_stop 100
 -> Validation loss : 1.1265
LR = 0.0001 | WD = 0.05
early_stop 104
 -> Validation loss : 1.1098
LR = 0.0001 | WD = 0.1
early_stop 104
 -> Validation loss : 1.1166
LR = 0.0001 | WD = 0.2
early_stop 103
 -> Validation loss : 1.1136

 Best LR : 0.0001 | Best WD : 0.05 (Val Avg Loss: 1.1098)
Test Accuracy : 0.6139

MODE: shared_context | HEADS: 64
LR = 0.0001 | WD = 0.0
early_stop 105
 -> Validation loss : 1.1147
LR = 0.0001 | WD = 0.05
early_stop 101
 -> Validation loss : 1.1157
LR = 0.0001 | WD = 0.1
early_stop 107
 -> Validation loss : 1.1

In [27]:
df_heads_pos_results

Num Heads (Patch 16),16,32,64,128,384
Mode,,,,,
shared_context,0.619221,0.61395,0.616788,0.628143,0.619627
independent_context,0.621249,0.623277,0.622871,0.622871,0.627737


In [28]:
df_heads_results

Num Heads (Patch 16),16,32,64,128,384
Mode,,,,,
shared_context,0.624899,0.623682,0.621249,0.620032,0.626926
independent_context,0.619221,0.618816,0.628548,0.624493,0.62571


In [29]:
PATCH = 8
MODES = ["shared_context", "independent_context"]
HEADS_TO_TEST = [16, 32, 64, 128, 384] 

df_heads_results = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_results.index.name = "Mode"
df_heads_results.columns.name = f"Num Heads (Patch {PATCH})"

tr_loader, va_loader, te_loader = get_dataloaders_for_patch(PATCH)

for mode in MODES:
    for heads in HEADS_TO_TEST:
        print(f"MODE: {mode} | HEADS: {heads}")
        
        _, test_acc = grid_search_attention_model(
            model_class=SingleScaleMultiHeadClassifier,
            model_kwargs={"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        
        df_heads_results.loc[mode, heads] = test_acc

MODE: shared_context | HEADS: 16
LR = 0.0001 | WD = 0.0
early_stop 110
 -> Validation loss : 1.1434
LR = 0.0001 | WD = 0.05
early_stop 125
 -> Validation loss : 1.1351
LR = 0.0001 | WD = 0.1
early_stop 111
 -> Validation loss : 1.1365
LR = 0.0001 | WD = 0.2
early_stop 109
 -> Validation loss : 1.1575

 Best LR : 0.0001 | Best WD : 0.05 (Val Avg Loss: 1.1351)
Test Accuracy : 0.6087

MODE: shared_context | HEADS: 32
LR = 0.0001 | WD = 0.0
early_stop 113
 -> Validation loss : 1.1380
LR = 0.0001 | WD = 0.05
early_stop 111
 -> Validation loss : 1.1378
LR = 0.0001 | WD = 0.1
early_stop 114
 -> Validation loss : 1.1378
LR = 0.0001 | WD = 0.2
early_stop 121
 -> Validation loss : 1.1332

 Best LR : 0.0001 | Best WD : 0.2 (Val Avg Loss: 1.1332)
Test Accuracy : 0.6123

MODE: shared_context | HEADS: 64
LR = 0.0001 | WD = 0.0
early_stop 117
 -> Validation loss : 1.1306
LR = 0.0001 | WD = 0.05
early_stop 112
 -> Validation loss : 1.1331
LR = 0.0001 | WD = 0.1
early_stop 119
 -> Validation loss : 1.1

In [30]:
df_heads_results

Num Heads (Patch 8),16,32,64,128,384
Mode,,,,,
shared_context,0.608678,0.612328,0.611517,0.611517,0.620032
independent_context,0.609895,0.6103,0.614761,0.611517,0.616788


TODO :
- target_dim?
- result diff de maxime car moi moi pytorch pour reglog

In [31]:
import torch
import torch.nn as nn

class CrossScaleMultiHeadClassifier(nn.Module):
    def __init__(
        self, num_vars, num_classes, num_patches_q, num_patches_kv, 
        in_features=384, num_heads=8
    ):
        super().__init__()
        self.num_vars = num_vars
        self.num_patches_q = num_patches_q
        self.num_patches_kv = num_patches_kv
        
        self.seq_len_q = num_vars * num_patches_q
        
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=in_features, num_heads=num_heads, dropout=0.1, batch_first=True
        )
        
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.Linear(num_vars * in_features, num_classes)

    def forward(self, x):
        # x est coupé, donc x_q et x_kv ne sont plus contigus en mémoire
        x_q = x[:, :self.seq_len_q, :]   
        x_kv = x[:, self.seq_len_q:, :]  
        
        B = x.shape[0]
        F = x.shape[-1]
        
        # --- ÉTAPE 1 : Préparation de la Query ---
        # Utilisation de reshape au lieu de view !
        x_q_reshaped = x_q.reshape(B, self.num_vars, self.num_patches_q, F)
        x_q_first = x_q_reshaped[:, :, 0:1, :] 
        seq_q = x_q_first.reshape(B * self.num_vars, 1, F) # <-- CORRIGÉ
        
        # --- ÉTAPE 2 : Préparation des Keys/Values ---
        # Utilisation de reshape au lieu de view !
        seq_kv = x_kv.reshape(B * self.num_vars, self.num_patches_kv, F) # <-- CORRIGÉ
        
        # --- ÉTAPE 3 : CROSS-ATTENTION ---
        fused_seq, _ = self.cross_attn(query=seq_q, key=seq_kv, value=seq_kv)
        
        fused_seq = fused_seq
        # --- ÉTAPE 4 : CLASSIFICATION ---
        pooled_flat = fused_seq.squeeze(1) 
        final_repr = pooled_flat.reshape(B, -1) # <-- CORRIGÉ par sécurité
        
        return self.linear(self.dropout(final_repr))

In [32]:
def get_dataloaders_cross_scale(patch_q=64, patch_kv=8, batch_size=256):
    # 💡 L'ASTUCE : On utilise torch.cat pour coller les séquences
    Z_train_combined = torch.cat([Z_train_patch[patch_q], Z_train_patch[patch_kv]], dim=1).cpu().numpy()
    Z_test_combined = torch.cat([Z_test_patch[patch_q], Z_test_patch[patch_kv]], dim=1).to(DEVICE)
    
    y_train_full = y_train_tensor.cpu().numpy()
    
    Z_train_split, Z_val_split, y_train_split, y_val_split = train_test_split(
        Z_train_combined, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
    )
    
    Z_tr = torch.tensor(Z_train_split, device=DEVICE)
    y_tr = torch.tensor(y_train_split, dtype=torch.long, device=DEVICE)
    Z_va = torch.tensor(Z_val_split, device=DEVICE)
    y_va = torch.tensor(y_val_split, dtype=torch.long, device=DEVICE)
    Z_te = Z_test_combined # Déjà sur DEVICE
    y_te = y_test_tensor.to(DEVICE)
    
    tr_loader = DataLoader(TensorDataset(Z_tr, y_tr), batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(TensorDataset(Z_va, y_va), batch_size=batch_size, shuffle=False)
    te_loader = DataLoader(TensorDataset(Z_te, y_te), batch_size=batch_size, shuffle=False)
    
    return tr_loader, va_loader, te_loader

In [33]:
PATCH_Q = 64
PATCH_KV = 8

# 1. On récupère les loaders "combinés"
tr_loader, va_loader, te_loader = get_dataloaders_cross_scale(PATCH_Q, PATCH_KV)

# 2. On calcule dynamiquement les tailles pour que le modèle sache où couper
num_p_q = Z_train_patch[PATCH_Q].shape[1] // NUM_VARS
num_p_kv = Z_train_patch[PATCH_KV].shape[1] // NUM_VARS

print(f"🚀 Lancement de la Cross-Scale Attention (Q={PATCH_Q}, KV={PATCH_KV})")

# 3. Appel de ton Grid Search habituel !
best_model, test_acc = grid_search_attention_model(
    model_class=CrossScaleMultiHeadClassifier,
    model_kwargs={
        "num_vars": NUM_VARS,
        "num_classes": num_classes,
        "num_patches_q": num_p_q,
        "num_patches_kv": num_p_kv,
        "num_heads": 16
    },
    train_loader=tr_loader, 
    val_loader=va_loader, 
    test_loader=te_loader, 
    device=DEVICE
)

print(f"✅ Terminé ! (Accuracy : {test_acc:.4f})")

🚀 Lancement de la Cross-Scale Attention (Q=64, KV=8)
LR = 0.0001 | WD = 0.0
early_stop 109
 -> Validation loss : 1.1130
LR = 0.0001 | WD = 0.05
early_stop 111
 -> Validation loss : 1.1092
LR = 0.0001 | WD = 0.1
early_stop 106
 -> Validation loss : 1.0993
LR = 0.0001 | WD = 0.2
early_stop 111
 -> Validation loss : 1.1031

 Best LR : 0.0001 | Best WD : 0.1 (Val Avg Loss: 1.0993)
Test Accuracy : 0.6249

✅ Terminé ! (Accuracy : 0.6249)


In [34]:
def get_all_scales_dataloaders(batch_size=256):
    # Ordre des résolutions : 64, 32, 16, 8
    Z_tr_comb = torch.cat([
        Z_train_patch[64], Z_train_patch[32], Z_train_patch[16], Z_train_patch[8]
    ], dim=1).cpu().numpy()
    
    Z_te_comb = torch.cat([
        Z_test_patch[64], Z_test_patch[32], Z_test_patch[16], Z_test_patch[8]
    ], dim=1).to(DEVICE)
    
    y_train_full = y_train_tensor.cpu().numpy()
    
    Z_train_split, Z_val_split, y_train_split, y_val_split = train_test_split(
        Z_tr_comb, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
    )
    
    Z_tr = torch.tensor(Z_train_split, device=DEVICE)
    y_tr = torch.tensor(y_train_split, dtype=torch.long, device=DEVICE)
    Z_va = torch.tensor(Z_val_split, device=DEVICE)
    y_va = torch.tensor(y_val_split, dtype=torch.long, device=DEVICE)
    Z_te = Z_te_comb
    y_te = y_test_tensor.to(DEVICE)
    
    tr_loader = DataLoader(TensorDataset(Z_tr, y_tr), batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(TensorDataset(Z_va, y_va), batch_size=batch_size, shuffle=False)
    te_loader = DataLoader(TensorDataset(Z_te, y_te), batch_size=batch_size, shuffle=False)
    
    return tr_loader, va_loader, te_loader

In [35]:
import torch
import torch.nn as nn

class SequentialCrossScaleClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, patches_counts, in_features=384, num_heads=8):
        super().__init__()
        self.num_vars = num_vars
        
        # patches_counts est un dictionnaire : {64: 2, 32: 3, 16: 4, 8: 6}
        self.p_counts = patches_counts
        
        # 3 couches d'attention distinctes pour apprendre les relations spécifiques à chaque échelle
        self.attn_32 = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
        self.attn_16 = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
        self.attn_8  = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
        
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.Linear(num_vars * in_features, num_classes)

    def forward(self, x):
        B = x.shape[0]
        F = x.shape[-1]
        
        # --- DÉCOUPAGE DYNAMIQUE DU TENSEUR ---
        start = 0
        x_dict = {}
        for scale in [64, 32, 16, 8]:
            length = self.num_vars * self.p_counts[scale]
            x_dict[scale] = x[:, start : start + length, :]
            start += length
            
        # --- PRÉPARATION ---
        # Query initiale : Le 1er élément du Patch 64
        x_64_r = x_dict[64].reshape(B, self.num_vars, self.p_counts[64], F)
        seq_q = x_64_r[:, :, 0:1, :].reshape(B * self.num_vars, 1, F)
        
        # Keys/Values : Les patchs complets
        seq_32 = x_dict[32].reshape(B * self.num_vars, self.p_counts[32], F)
        seq_16 = x_dict[16].reshape(B * self.num_vars, self.p_counts[16], F)
        seq_8  = x_dict[8].reshape(B * self.num_vars, self.p_counts[8], F)
        
        # --- CASCADE D'ATTENTION ---
        # Étape 1 : 64 interroge 32
        q1, _ = self.attn_32(query=seq_q, key=seq_32, value=seq_32)
        q1 = seq_q + q1 # Résiduel
        
        # Étape 2 : Le résultat interroge 16
        q2, _ = self.attn_16(query=q1, key=seq_16, value=seq_16)
        q2 = q1 + q2 # Résiduel
        
        # Étape 3 : Le résultat interroge 8
        q3, _ = self.attn_8(query=q2, key=seq_8, value=seq_8)
        fused_seq = q2 + q3 # Résiduel final
        
        # --- CLASSIFICATION ---
        final_repr = fused_seq.squeeze(1).reshape(B, -1)
        return self.linear(self.dropout(final_repr))

In [36]:
class ConcatCrossScaleClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, patches_counts, in_features=384, num_heads=8):
        super().__init__()
        self.num_vars = num_vars
        self.p_counts = patches_counts
        
        # Une seule couche d'attention
        self.cross_attn = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
        
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.Linear(num_vars * in_features, num_classes)

    def forward(self, x):
        B = x.shape[0]
        F = x.shape[-1]
        
        # --- DÉCOUPAGE DYNAMIQUE ---
        start = 0
        x_dict = {}
        for scale in [64, 32, 16, 8]:
            length = self.num_vars * self.p_counts[scale]
            x_dict[scale] = x[:, start : start + length, :]
            start += length
            
        # --- PRÉPARATION DE LA QUERY ---
        x_64_r = x_dict[64].reshape(B, self.num_vars, self.p_counts[64], F)
        seq_q = x_64_r[:, :, 0:1, :].reshape(B * self.num_vars, 1, F)
        
        # --- CONCATÉNATION DES KEYS/VALUES (32 + 16 + 8) ---
        # On reshape d'abord pour séparer les variables (pour ne pas mélanger les variables entre elles !)
        x_32_r = x_dict[32].reshape(B, self.num_vars, self.p_counts[32], F)
        x_16_r = x_dict[16].reshape(B, self.num_vars, self.p_counts[16], F)
        x_8_r  = x_dict[8].reshape(B, self.num_vars, self.p_counts[8], F)
        
        # On concatène sur la dimension des patchs (dim=2)
        kv_combined = torch.cat([x_32_r, x_16_r, x_8_r], dim=2) 
        seq_kv = kv_combined.reshape(B * self.num_vars, -1, F)
        
        # --- CROSS-ATTENTION GLOBALE ---
        fused_seq, _ = self.cross_attn(query=seq_q, key=seq_kv, value=seq_kv)
        fused_seq = seq_q + fused_seq # Résiduel
        
        # --- CLASSIFICATION ---
        final_repr = fused_seq.squeeze(1).reshape(B, -1)
        return self.linear(self.dropout(final_repr))

In [37]:
# 1. On récupère les DataLoaders avec toutes les résolutions
tr_loader, va_loader, te_loader = get_all_scales_dataloaders()

# 2. On calcule automatiquement combien de patchs on a par échelle
patches_counts = {
    64: Z_train_patch[64].shape[1] // NUM_VARS,
    32: Z_train_patch[32].shape[1] // NUM_VARS,
    16: Z_train_patch[16].shape[1] // NUM_VARS,
    8:  Z_train_patch[8].shape[1] // NUM_VARS
}

print(f"Tailles des patchs calculées : {patches_counts}\n")

# --- TEST 1 : CASCADE ---
print("🚀 Lancement : APPROCHE SÉQUENTIELLE (Cascade)")
best_model_seq, test_acc_seq = grid_search_attention_model(
    model_class=SequentialCrossScaleClassifier,
    model_kwargs={
        "num_vars": NUM_VARS,
        "num_classes": num_classes,
        "patches_counts": patches_counts,
        "num_heads": 8
    },
    train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
)

# --- TEST 2 : CONCATÉNATION ---
print("🚀 Lancement : APPROCHE CONCATÉNÉE (Globale)")
best_model_concat, test_acc_concat = grid_search_attention_model(
    model_class=ConcatCrossScaleClassifier,
    model_kwargs={
        "num_vars": NUM_VARS,
        "num_classes": num_classes,
        "patches_counts": patches_counts,
        "num_heads": 8
    },
    train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
)

print("\n" + "="*40)
print("🏆 RÉSULTATS FINAUX 🏆")
print("="*40)
print(f"Précision Approche Séquentielle : {test_acc_seq:.4f}")
print(f"Précision Approche Concaténée   : {test_acc_concat:.4f}")

Tailles des patchs calculées : {64: 2, 32: 3, 16: 4, 8: 6}

🚀 Lancement : APPROCHE SÉQUENTIELLE (Cascade)
LR = 0.0001 | WD = 0.0
early_stop 84
 -> Validation loss : 1.0983
LR = 0.0001 | WD = 0.05
early_stop 89
 -> Validation loss : 1.1123
LR = 0.0001 | WD = 0.1
early_stop 88
 -> Validation loss : 1.0919
LR = 0.0001 | WD = 0.2
early_stop 86
 -> Validation loss : 1.1021

 Best LR : 0.0001 | Best WD : 0.1 (Val Avg Loss: 1.0919)
Test Accuracy : 0.6367

🚀 Lancement : APPROCHE CONCATÉNÉE (Globale)
LR = 0.0001 | WD = 0.0
early_stop 105
 -> Validation loss : 1.0691
LR = 0.0001 | WD = 0.05
early_stop 99
 -> Validation loss : 1.0764
LR = 0.0001 | WD = 0.1
early_stop 105
 -> Validation loss : 1.0973
LR = 0.0001 | WD = 0.2
early_stop 99
 -> Validation loss : 1.0911

 Best LR : 0.0001 | Best WD : 0.0 (Val Avg Loss: 1.0691)
Test Accuracy : 0.6395


🏆 RÉSULTATS FINAUX 🏆
Précision Approche Séquentielle : 0.6367
Précision Approche Concaténée   : 0.6395


In [38]:
import torch
import torch.nn as nn

class SequentialCrossScaleClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, patches_counts, in_features=384, num_heads=8, shared_layer=False):
        super().__init__()
        self.num_vars = num_vars
        self.p_counts = patches_counts
        self.shared_layer = shared_layer
        
        # --- CHOIX DU PARTAGE DE POIDS ---
        if shared_layer:
            # Une seule couche qui fera les 3 étapes
            self.shared_attn = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
        else:
            # 3 couches indépendantes expertes de leur échelle
            self.attn_32 = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
            self.attn_16 = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
            self.attn_8  = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
        
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.Linear(num_vars * in_features, num_classes)

    def forward(self, x):
        B = x.shape[0]
        F = x.shape[-1]
        
        start = 0
        x_dict = {}
        for scale in [64, 32, 16, 8]:
            length = self.num_vars * self.p_counts[scale]
            x_dict[scale] = x[:, start : start + length, :]
            start += length
            
        # Préparation Query et KV
        x_64_r = x_dict[64].reshape(B, self.num_vars, self.p_counts[64], F)
        seq_q = x_64_r[:, :, 0:1, :].reshape(B * self.num_vars, 1, F)
        
        seq_32 = x_dict[32].reshape(B * self.num_vars, self.p_counts[32], F)
        seq_16 = x_dict[16].reshape(B * self.num_vars, self.p_counts[16], F)
        seq_8  = x_dict[8].reshape(B * self.num_vars, self.p_counts[8], F)
        
        # Attribution des couches
        a32 = self.shared_attn if self.shared_layer else self.attn_32
        a16 = self.shared_attn if self.shared_layer else self.attn_16
        a8  = self.shared_attn if self.shared_layer else self.attn_8
        
        # Cascade
        q1, _ = a32(query=seq_q, key=seq_32, value=seq_32)
        q1 = seq_q + q1 
        
        q2, _ = a16(query=q1, key=seq_16, value=seq_16)
        q2 = q1 + q2 
        
        q3, _ = a8(query=q2, key=seq_8, value=seq_8)
        fused_seq = q2 + q3 
        
        final_repr = fused_seq.squeeze(1).reshape(B, -1)
        return self.linear(self.dropout(final_repr))

In [39]:
class ParallelCrossScaleClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, patches_counts, in_features=384, num_heads=8, shared_layer=True):
        super().__init__()
        self.num_vars = num_vars
        self.p_counts = patches_counts
        self.shared_layer = shared_layer
        
        if shared_layer:
            self.shared_attn = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
        else:
            self.attn_32 = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
            self.attn_16 = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
            self.attn_8  = nn.MultiheadAttention(in_features, num_heads, dropout=0.1, batch_first=True)
        
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.Linear(num_vars * in_features, num_classes)

    def forward(self, x):
        B = x.shape[0]
        F = x.shape[-1]
        
        start = 0
        x_dict = {}
        for scale in [64, 32, 16, 8]:
            length = self.num_vars * self.p_counts[scale]
            x_dict[scale] = x[:, start : start + length, :]
            start += length
            
        x_64_r = x_dict[64].reshape(B, self.num_vars, self.p_counts[64], F)
        seq_q = x_64_r[:, :, 0:1, :].reshape(B * self.num_vars, 1, F)
        
        x_32_r = x_dict[32].reshape(B, self.num_vars, self.p_counts[32], F)
        x_16_r = x_dict[16].reshape(B, self.num_vars, self.p_counts[16], F)
        x_8_r  = x_dict[8].reshape(B, self.num_vars, self.p_counts[8], F)
        
        if self.shared_layer:
            # MODE CONCATÉNÉ : On colle tout et on interroge 1 fois
            kv_combined = torch.cat([x_32_r, x_16_r, x_8_r], dim=2) 
            seq_kv = kv_combined.reshape(B * self.num_vars, -1, F)
            fused_seq, _ = self.shared_attn(query=seq_q, key=seq_kv, value=seq_kv)
            fused_seq = seq_q + fused_seq
        else:
            # MODE PARALLÈLE : Interrogation indépendante et addition
            seq_32 = x_32_r.reshape(B * self.num_vars, self.p_counts[32], F)
            seq_16 = x_16_r.reshape(B * self.num_vars, self.p_counts[16], F)
            seq_8  = x_8_r.reshape(B * self.num_vars, self.p_counts[8], F)
            
            out_32, _ = self.attn_32(query=seq_q, key=seq_32, value=seq_32)
            out_16, _ = self.attn_16(query=seq_q, key=seq_16, value=seq_16)
            out_8, _  = self.attn_8(query=seq_q, key=seq_8, value=seq_8)
            
            fused_seq = seq_q + out_32 + out_16 + out_8
            
        final_repr = fused_seq.squeeze(1).reshape(B, -1)
        return self.linear(self.dropout(final_repr))

In [40]:
import pandas as pd

# 1. Obtenir le DataLoader Multi-Échelles et le dictionnaire de tailles
tr_loader, va_loader, te_loader = get_all_scales_dataloaders()

patches_counts = {
    64: Z_train_patch[64].shape[1] // NUM_VARS,
    32: Z_train_patch[32].shape[1] // NUM_VARS,
    16: Z_train_patch[16].shape[1] // NUM_VARS,
    8:  Z_train_patch[8].shape[1] // NUM_VARS
}

# 2. Définition des 4 expériences
experiences = [
    ("Séquentiel (Cascade)", SequentialCrossScaleClassifier, True),   # Shared
    ("Séquentiel (Cascade)", SequentialCrossScaleClassifier, False),  # 3 Perso
    ("Parallèle (Concat)", ParallelCrossScaleClassifier, True),       # Shared
    ("Parallèle (Concat)", ParallelCrossScaleClassifier, False)       # 3 Perso
]

results = pd.DataFrame(index=["Séquentiel (Cascade)", "Parallèle (Concat)"], 
                       columns=["1 Couche Partagée", "3 Couches Perso"])

# 3. Lancement de la grille
print("🚀 DÉBUT DE L'ÉTUDE D'ABLATION MULTI-ÉCHELLES\n")

for arch_name, model_class, shared in experiences:
    col_name = "1 Couche Partagée" if shared else "3 Couches Perso"
    print(f"🔄 Test en cours : {arch_name} | {col_name}")
    
    _, test_acc = grid_search_attention_model(
        model_class=model_class,
        model_kwargs={
            "num_vars": NUM_VARS,
            "num_classes": num_classes,
            "patches_counts": patches_counts,
            "num_heads": 16,
            "shared_layer": shared
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
    )
    
    results.loc[arch_name, col_name] = test_acc

print("\n" + "="*50)
print("🏆 RÉSULTATS FINAUX (Accuracy Test) 🏆")
print("="*50)
display(results.astype(float).round(4))

🚀 DÉBUT DE L'ÉTUDE D'ABLATION MULTI-ÉCHELLES

🔄 Test en cours : Séquentiel (Cascade) | 1 Couche Partagée
LR = 0.0001 | WD = 0.0
early_stop 90
 -> Validation loss : 1.1195
LR = 0.0001 | WD = 0.05
early_stop 96
 -> Validation loss : 1.1125
LR = 0.0001 | WD = 0.1
early_stop 97
 -> Validation loss : 1.1208
LR = 0.0001 | WD = 0.2
early_stop 95
 -> Validation loss : 1.1204

 Best LR : 0.0001 | Best WD : 0.05 (Val Avg Loss: 1.1125)
Test Accuracy : 0.6237

🔄 Test en cours : Séquentiel (Cascade) | 3 Couches Perso
LR = 0.0001 | WD = 0.0
early_stop 89
 -> Validation loss : 1.0728
LR = 0.0001 | WD = 0.05
early_stop 83
 -> Validation loss : 1.1020
LR = 0.0001 | WD = 0.1
early_stop 88
 -> Validation loss : 1.0943
LR = 0.0001 | WD = 0.2
early_stop 86
 -> Validation loss : 1.1019

 Best LR : 0.0001 | Best WD : 0.0 (Val Avg Loss: 1.0728)
Test Accuracy : 0.6427

🔄 Test en cours : Parallèle (Concat) | 1 Couche Partagée
LR = 0.0001 | WD = 0.0
early_stop 100
 -> Validation loss : 1.0812
LR = 0.0001 | WD = 

,1 Couche Partagée,3 Couches Perso
Séquentiel (Cascade),0.6237,0.6427
Parallèle (Concat),0.6346,0.6529


In [41]:
import pandas as pd

# 1. Obtenir le DataLoader Multi-Échelles et le dictionnaire de tailles
tr_loader, va_loader, te_loader = get_all_scales_dataloaders()

patches_counts = {
    64: Z_train_patch[64].shape[1] // NUM_VARS,
    32: Z_train_patch[32].shape[1] // NUM_VARS,
    16: Z_train_patch[16].shape[1] // NUM_VARS,
    8:  Z_train_patch[8].shape[1] // NUM_VARS
}

# 2. Définition des 4 expériences
experiences = [
    ("Séquentiel (Cascade)", SequentialCrossScaleClassifier, True),   # Shared
    ("Séquentiel (Cascade)", SequentialCrossScaleClassifier, False),  # 3 Perso
    ("Parallèle (Concat)", ParallelCrossScaleClassifier, True),       # Shared
    ("Parallèle (Concat)", ParallelCrossScaleClassifier, False)       # 3 Perso
]

results = pd.DataFrame(index=["Séquentiel (Cascade)", "Parallèle (Concat)"], 
                       columns=["1 Couche Partagée", "3 Couches Perso"])

# 3. Lancement de la grille
print("🚀 DÉBUT DE L'ÉTUDE D'ABLATION MULTI-ÉCHELLES\n")

for arch_name, model_class, shared in experiences:
    col_name = "1 Couche Partagée" if shared else "3 Couches Perso"
    print(f"🔄 Test en cours : {arch_name} | {col_name}")
    
    _, test_acc = grid_search_attention_model(
        model_class=model_class,
        model_kwargs={
            "num_vars": NUM_VARS,
            "num_classes": num_classes,
            "patches_counts": patches_counts,
            "num_heads": 64,
            "shared_layer": shared
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
    )
    
    results.loc[arch_name, col_name] = test_acc

print("\n" + "="*50)
print("🏆 RÉSULTATS FINAUX (Accuracy Test) 🏆")
print("="*50)
display(results.astype(float).round(4))

🚀 DÉBUT DE L'ÉTUDE D'ABLATION MULTI-ÉCHELLES

🔄 Test en cours : Séquentiel (Cascade) | 1 Couche Partagée
LR = 0.0001 | WD = 0.0
early_stop 100
 -> Validation loss : 1.0916
LR = 0.0001 | WD = 0.05
early_stop 94
 -> Validation loss : 1.0985
LR = 0.0001 | WD = 0.1
early_stop 97
 -> Validation loss : 1.1033
LR = 0.0001 | WD = 0.2
early_stop 99
 -> Validation loss : 1.1079

 Best LR : 0.0001 | Best WD : 0.0 (Val Avg Loss: 1.0916)
Test Accuracy : 0.6281

🔄 Test en cours : Séquentiel (Cascade) | 3 Couches Perso
LR = 0.0001 | WD = 0.0
early_stop 90
 -> Validation loss : 1.0813
LR = 0.0001 | WD = 0.05
early_stop 92
 -> Validation loss : 1.0706
LR = 0.0001 | WD = 0.1
early_stop 89
 -> Validation loss : 1.0677
LR = 0.0001 | WD = 0.2
early_stop 91
 -> Validation loss : 1.0850

 Best LR : 0.0001 | Best WD : 0.1 (Val Avg Loss: 1.0677)
Test Accuracy : 0.6444

🔄 Test en cours : Parallèle (Concat) | 1 Couche Partagée
LR = 0.0001 | WD = 0.0
early_stop 109
 -> Validation loss : 1.0688
LR = 0.0001 | WD = 

,1 Couche Partagée,3 Couches Perso
Séquentiel (Cascade),0.6281,0.6444
Parallèle (Concat),0.6403,0.6448
